In [ ]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [ ]:
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

In [ ]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)
x_totem, y_totem, yerr_totem = process_data(totem_data, totem_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

x_7_totem, y_7_totem, yerr_7_totem = x_totem[0], y_totem[0], yerr_totem[0]
x_8_totem, y_8_totem, yerr_8_totem = x_totem[1], y_totem[1], yerr_totem[1]
x_13_totem, y_13_totem, yerr_13_totem = x_totem[2], y_totem[2], yerr_totem[2]

In [ ]:

b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_log_atlas = ensemble_parameters[ensemble_atlas][log_model_type]
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_log_atlas, initial_params_low_log_atlas, initial_params_high_log_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, log_model_type)

initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)

# Para Totem
initial_params_log_totem, initial_params_low_log_totem, initial_params_high_log_totem = \
    get_parameters_with_variations(ensemble_parameters, ensemble_totem, log_model_type)

initial_params_pl_totem, initial_params_low_pl_totem, initial_params_high_pl_totem = \
    get_parameters_with_variations(ensemble_parameters, ensemble_totem, pl_model_type)



In [ ]:
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [ ]:
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        def inner_integral(x_inner):
            return fixed_quad(
                lambda y: integrand(y, x_inner, params['mg'], params['a1'], 
                                  params['a2'], m2, q2, sqrt_s),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [ ]:
def create_least_squares(x, y, yerr, sqrt_s, model_type):
    return LeastSquares(x, y, yerr, 
        lambda x, eps, mg, a1, a2: model_function(x, eps, mg, a1, a2, sqrt_s, model_type))

def total_cost(dataset, model_type):
    return sum([
        create_least_squares(*dataset[7000], 7000, model_type),
        create_least_squares(*dataset[8000], 8000, model_type),
        create_least_squares(*dataset[13000], 13000, model_type)
    ])

# Estrutura dos dados por experimento
atlas_data = {
    7000: (x_7_atlas, y_7_atlas, yerr_7_atlas),
    8000: (x_8_atlas, y_8_atlas, yerr_8_atlas),
    13000: (x_13_atlas, y_13_atlas, yerr_13_atlas)
}

totem_data = {
    7000: (x_7_totem, y_7_totem, yerr_7_totem),
    8000: (x_8_totem, y_8_totem, yerr_8_totem),
    13000: (x_13_totem, y_13_totem, yerr_13_totem)
}

# Custos totais
total_cost_log_atlas = total_cost(atlas_data, 'log')
total_cost_pl_atlas = total_cost(atlas_data, 'pl')
total_cost_log_totem = total_cost(totem_data, 'log')
total_cost_pl_totem = total_cost(totem_data, 'pl')


In [ ]:
# def otimization(total_cost_func, initial_params, initial_params_low, initial_params_high, model_type: str, ensemble: str):
#     print('\n')
#     print(80 * '-')
#     print(f"Iniciando otimização dos parâmetros usando LeastSquares para {model_type} em {ensemble.upper()}")

#     start_time = time.time()
    
#     m = Minuit(total_cost_func, 
#                eps=initial_params['epsilon'],
#                mg=initial_params['mg'],
#                a1=initial_params['a1'],
#                a2=initial_params['a2'])

#     if model_type == 'pl':
#         # Configurações adicionais
#         down = 0.97
#         up = 2- down

#         m.limits['mg'] = (down * initial_params['mg'], up *initial_params['mg'])
#         m.limits['eps'] = (down * initial_params['epsilon'], up *initial_params['epsilon'])
#         # m.limits['a2'] = (initial_params_low['a2'], initial_params_high['a2'])

#         m.limits['a1'] = (initial_params_low['a1'], initial_params_high['a1'])

#         # Configurações adicionais
#         m.strategy = 2
#         m.errordef = 7.79
#         m.tol = 1e-2
        
#         m.migrad(ncall=250)
#         m.hesse(ncall=20)

#     else:
#         down = 0.99
#         up = 2- down

#         m.limits['mg'] = (down * initial_params['mg'], up * initial_params['mg'])
#         m.limits['eps'] = (down * initial_params['epsilon'], up * initial_params['epsilon'])
#         m.limits['a2'] = (down * initial_params['a2'], up * initial_params['a2'])
#         # m.limits['a1'] = (down * initial_params['a1'], up * initial_params['a1'])

#         # Configurações adicionais
#         m.strategy = 2
#         m.errordef = 7.79
#         m.tol = 1e-2
         
#         m.migrad(ncall=85)
#         m.hesse(ncall=10)
            

#     print(f'Finalizado a minimização para {model_type} em {ensemble}')

#     execution_time = time.time() - start_time
    
#     minutes = int(execution_time // 60)
#     seconds = execution_time % 60
#     print(f'Tempo de execução para {model_type} em {ensemble}: {minutes} min {seconds:.2f} s \n')
    
#     print(f"Parâmetros otimizados para {model_type} em {ensemble}: \n")
#     print(f'mg: {m.values["mg"]} ± {m.errors["mg"]}')
#     print(f'eps: {m.values["eps"]} ± {m.errors["eps"]}')
#     print(f'a1: {m.values["a1"]} ± {m.errors["a1"]}')
#     print(f'a2: {m.values["a2"]} ± {m.errors["a2"]}')
#     print(f'chi2/ndof: {m.fval / m.ndof}')


#     return m




In [ ]:
def otimization(total_cost_func, initial_params, initial_params_low, initial_params_high, model_type: str, ensemble: str):
    print('\n')
    print(80 * '-')
    print(f"Iniciando otimização dos parâmetros usando LeastSquares para {model_type} em {ensemble.upper()}")

    start_time = time.time()
    
    m = Minuit(total_cost_func, 
               eps=initial_params['epsilon'],
               mg=initial_params['mg'],
               a1=initial_params['a1'],
               a2=initial_params['a2'])

    if model_type == 'pl':
        # Configurações adicionais
        down = 0.91
        up = 2- down

        # m.limits['mg'] = (down * initial_params['mg'], up *initial_params['mg'])
        # m.limits['eps'] = (down * initial_params['epsilon'], up *initial_params['epsilon'])
        # # m.limits['a2'] = (initial_params_low['a2'], initial_params_high['a2'])

        m.limits['a1'] = (down * initial_params['a1'], up * initial_params['a1'])

        # Configurações adicionais
        m.strategy = 0
        m.errordef = 7.79
        m.tol = 1e-2
        
        m.migrad(ncall=70)
        m.hesse(ncall=20)


    else:
        down = 0.94
        up = 2- down

        m.limits['mg'] = (down * initial_params['mg'], up * initial_params['mg'])
        # m.limits['eps'] = (down * initial_params['epsilon'], up * initial_params['epsilon'])
        m.limits['a2'] = (down * initial_params['a2'], up * initial_params['a2'])
        # m.limits['a1'] = (down * initial_params['a1'], up * initial_params['a1'])

        # Configurações adicionais
        m.strategy = 2
        m.errordef = 7.79
        m.tol = 1e-2
         
        m.migrad(ncall=70)
        m.hesse(ncall=10)

    print(f'Finalizado a minimização para {model_type} em {ensemble}')

    execution_time = time.time() - start_time
    
    minutes = int(execution_time // 60)
    seconds = execution_time % 60
    print(f'Tempo de execução para {model_type} em {ensemble}: {minutes} min {seconds:.2f} s \n')
    
    print(f"Parâmetros otimizados para {model_type} em {ensemble}: \n")
    print(f'mg: {m.values["mg"]} ± {m.errors["mg"]}')
    print(f'eps: {m.values["eps"]} ± {m.errors["eps"]}')
    print(f'a1: {m.values["a1"]} ± {m.errors["a1"]}')
    print(f'a2: {m.values["a2"]} ± {m.errors["a2"]}')
    print(f'chi2/ndof: {m.fval / m.ndof}')


    return m



In [ ]:


m_pl_atlas = otimization(total_cost_pl_atlas,
                         initial_params_pl_atlas,
                         initial_params_low_pl_atlas,
                         initial_params_high_pl_atlas, 'pl', 'atlas')



In [ ]:
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001
    n_points = 10000

    def integrand(y, x, mg, a1, a2, m2_func, q_val):
        k        = sqrt_s * x
        phi      = 2*np.pi*y
        jacobian = 2*np.pi*sqrt_s
        return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
                  T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2

        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, mg_model, q2),
                0, 1, n=n_points
            )[0]

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        diff_T = integral_value
        print(f"q2: {q2}, diff_T: {diff_T}")

        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}


In [ ]:

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    m_pl_atlas.values['eps'],
    m_pl_atlas.values['mg'], 
    m_pl_atlas.values['a1'],
    m_pl_atlas.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


In [ ]:
def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))



In [ ]:
# fig_atlas = go.Figure()


# # for pl atlas
# add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

# #-----------------------------------------------------------------------------------------------

# #-----------------------------------------------------------------------------------------------

# #data points
# add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# # Atualiza layout
# fig_atlas.update_layout(
#     title='dσ/dt vs. |t| - Log and PL models in ATLAS',
#     xaxis_title='|t| (GeV²)',
#     yaxis_title='dσ/dt (mb/GeV²)',
#     yaxis_type='log',
#     legend_title='Mass Model',
#     plot_bgcolor='white',
#     hovermode='x unified'
# )

# fig_atlas.update_xaxes(gridcolor='lightgray')
# fig_atlas.update_yaxes(gridcolor='lightgray')

# # fig_atlas.show(renderer='browser')


In [ ]:
data_sigma_tot_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

data_sigma_tot_totem = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_totem.dat",  # Supondo que existe um arquivo similar para TOTEM
    delim_whitespace=True,
    header=None,
    nrows=80
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

x_sigma_tot_totem = data_sigma_tot_totem[0].to_numpy()
y_sigma_tot_totem = data_sigma_tot_totem[1].to_numpy()
y_error_sigma_tot_totem = data_sigma_tot_totem[2].to_numpy()

start_sqrt_s = 1
max_sqrt_s = 13010
step = 100

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))

def get_sigma_tot(epsilon, mg, a1, a2, mg_model):

    lst_sigma_tot = []
    lst_sqrt_s = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s:
        def integrand(y, x, mg, a1, a2, m2_func):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s

            return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian
        
        def amp_calculation(diff_T, s, epsilon):
            alpha_pomeron = 1.0 + epsilon
            regge_factor = (s / s0) ** alpha_pomeron
            
            return 1j * 8.0 * regge_factor * diff_T

        s = sqrt_s ** 2

        integral_value = fixed_quad(
            lambda x: fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, mg_model),
                0, 1, n=n_points)[0],

            0, 1, n=n_points)[0]
        
        lst_sigma_tot.append(sigma_tot(
            amp_calculation(integral_value, s, epsilon), s))
        
        lst_sqrt_s.append(sqrt_s)
        sqrt_s += step
    return lst_sigma_tot, lst_sqrt_s


In [ ]:
#-----------------------------------------------------------------------------------------------

sigma_tot_pl_atlas = get_sigma_tot(
    m_pl_atlas.values['eps'],
    m_pl_atlas.values['mg'],
    m_pl_atlas.values['a1'],
    m_pl_atlas.values['a2'],
    m2_pl
)

sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
lst_sqrt_s = sigma_tot_pl_atlas[1]


# fig = go.Figure()

# add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

# #-----------------------------------------------------------------------------------------------
# #----

# add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')
# add_data_trace(fig, x_sigma_tot_totem, y_sigma_tot_totem, y_error_sigma_tot_totem, name='TOTEM', show_label=True, mode='markers')

# fig.update_layout(
#     title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
#     xaxis=dict(
#         title='√s [GeV]',
#         type='log',
#         range=[np.log10(2000), np.log10(14000)],
#     ),
#     yaxis=dict(
#         title='σ_tot [mb]',
#         range=[80, 125]
#     ),
#     showlegend=True,
#     legend=dict(
#         title='Ensembles'
#     ),
#     plot_bgcolor='white',
#     hovermode='x unified'
# )
    
# fig.update_xaxes(gridcolor='lightgray')
# fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")

In [ ]:
lst_q_integration = np.linspace(0, 0.2, 100)
lst_b_integration = np.linspace(0, 30, 100)

In [ ]:
# def model_function(x_born, eps, mg, a1, a2, sqrt_s, model_type='log'):

#     s = sqrt_s ** 2
#     results = []    

#     # loop sobre os pontos experimentais (para cada t)
#     for q_exp in x_born:

#         eik_amp_sum = 0 

#         for b_value in lst_b_integration:
#             chi_sum = 0
            
#             for q_int in lst_q_integration:

#                 def integrand(y, x, mg, a1, a2, m2_func, q_val):
#                     k        = sqrt_s * x
#                     phi      = 2*np.pi*y
#                     jacobian = 2*np.pi*sqrt_s
#                     return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
#                             T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian
#                 def inner_integral(x):
#                             return fixed_quad(
#                                 lambda y: integrand(y, x, mg, a1, a2, m2_pl, q_int),
#                                 0, 1, n=n_points
#                             )[0]

#                 integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
#                 diff_t = integral_value

#                 # diff_t = compute_k_phi_integral(mg, a1, a2, m2_pl, q_int, sqrt_s)
                
#                 t = -(q_int**2)
#                 # born_amp_value = born_amp(diff_t, s, eps, t)
#                 born_amp_value = amp_calculation(diff_t, sqrt_s**2, ensemble_parameters['atlas']['pl']['epsilon'], t)
#                 chi_value = (q_int * j0(b_value * q_int) * born_amp_value) / s

#                 # chi_value = chi_integral(sqrt_s, b_value, q_int, mg, a1, a2, m2_pl, eps)
                
#                 chi_sum += chi_value   

#             factor = 1 - np.exp(1j * chi_sum)
#             eik_amp_value = b_value * j0(b_value * q_exp) * factor * (1j * s)
#             eik_amp_sum += eik_amp_value
            
#         # eik_amp_value = eik_amp(s, 30, q_exp, chi_value)

#         # seção de choque diferencial
#         # diff_sigma = born_dif_sigma(eik_amp_sum, s)
#         diff_sigma = differential_sigma(eik_amp_sum, s)
#         results.append(diff_sigma)

#     return np.array(results)


In [ ]:
import numpy as np
# suponho que fixed_quad, T_1, T_2, amp_calculation, differential_sigma, j0,
# ensemble_parameters, m2_pl, lst_b_integration, lst_q_integration, n_points
# já estejam disponíveis no escopo onde esse código será usado.

def _integrand_yx(y, x, sqrt_s, mg, a1, a2, m2_func, q_val):
    """Integrando usado pelo fixed_quad: recebe y e x (escala k = sqrt_s * x)."""
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian

def _inner_integral_over_y(x, sqrt_s, mg, a1, a2, m2_func, q_val):
    """
    Integral em y para um dado x: ∫_0^1 integrand(y,x) dy,
    retornando apenas o valor (fixed_quad devolve (valor, err), pegamos [0]).
    """
    return fixed_quad(
        lambda y: _integrand_yx(y, x, sqrt_s, mg, a1, a2, m2_func, q_val),
        0, 1, n=n_points
    )[0]

def compute_diff_t_for_qint(q_int, sqrt_s, mg, a1, a2, m2_func):
    """
    Calcula o integral duplo aninhado que no código original virava 'diff_t':
      diff_t = ∫_0^1 dx [ ∫_0^1 dy integrand(y,x) ]
    Mantém fixed_quad aninhado exatamente como antes.
    """
    return fixed_quad(
        lambda x: _inner_integral_over_y(x, sqrt_s, mg, a1, a2, m2_func, q_int),
        0, 1, n=n_points
    )[0]

def model_function(x_born, eps, mg, a1, a2, sqrt_s, model_type='log'):
    """
    Versão reestruturada: mesmas entradas e mesmo fluxo numérico,
    com funções auxiliares externas aos loops para facilitar vetorização futura.
    Retorna np.array(results) com os mesmos valores do código original.
    """
    s = sqrt_s ** 2
    results = []

    # pré-calcula fator que é constante dentro do loop principal (micro otimização)
    factor_i_s = 1j * s

    # Nota: mantive o uso de ensemble_parameters['atlas']['pl']['epsilon']
    # exatamente como no seu código original para garantir igualdade numérica.

    for q_exp in x_born:
        eik_amp_sum = 0

        for b_value in lst_b_integration:
            chi_sum = 0

            for q_int in lst_q_integration:
                # calcula diff_t exatamente como antes (fixed_quad aninhado)
                diff_t = compute_diff_t_for_qint(q_int, sqrt_s, mg, a1, a2, m2_pl)

                t = -(q_int ** 2)

                # uso exatamente a mesma chamada a amp_calculation que no original
                born_amp_value = amp_calculation(
                    diff_t,
                    sqrt_s**2,
                    ensemble_parameters['atlas']['pl']['epsilon'],
                    t
                )

                chi_value = (q_int * j0(b_value * q_int) * born_amp_value) / s
                chi_sum += chi_value

            # fator e soma final do amplitude eikonal (mesma expressão)
            factor = 1 - np.exp(1j * chi_sum)
            eik_amp_value = b_value * j0(b_value * q_exp) * factor * factor_i_s
            eik_amp_sum += eik_amp_value

        # seção de choque diferencial (mesma chamada)
        diff_sigma = differential_sigma(eik_amp_sum, s)
        results.append(diff_sigma)

    return np.array(results)


In [ ]:
def make_least_squares(x, y, yerr, energy, model_type):
    return LeastSquares(x, y, yerr, 
        lambda x, eps, mg, a1, a2: model_function(x, eps, mg, a1, a2, energy, model_type))

total_cost_pl_atlas = (
    make_least_squares(x_7_atlas, y_7_atlas, yerr_7_atlas, 7000, 'pl')
)


In [ ]:
def otimization(total_cost_func, mg_init, eps_init, a1_init, a2_init, model_type: str, ensemble: str):
    print('\n')
    print(80 * '-')
    print(f"Iniciando otimização dos parâmetros usando LeastSquares para {model_type} em {ensemble.upper()}")

    m = Minuit(total_cost_func, 
               eps=eps_init,
               mg=mg_init,
               a1=a1_init,
               a2=a2_init)

    m.strategy = 2
    # m.errordef = 1
    # m.tol = 0.1


    m.simplex()
    m.simplex()

    m.migrad()
    m.migrad()
    m.migrad()

    if m.valid == False:
        while m.valid == False:
            print(f'Minuit inválido, migrad novamente tentativa {count+1}')
            m.migrad()
            count += 1
            if count >= 10:
                print(f'Minuit inválido após {count} tentativas.')
                break
    else:
        print("Minuit validado com sucesso!")

    m.hesse()
    # m.minos(cl = 0.9)

    print(f"Parâmetros otimizados para {model_type} em {ensemble}: \n")
    print(f'mg: {m.values["mg"]} ± {m.errors["mg"]}')
    print(f'eps: {m.values["eps"]} ± {m.errors["eps"]}')
    print(f'a1: {m.values["a1"]} ± {m.errors["a1"]}')
    print(f'a2: {m.values["a2"]} ± {m.errors["a2"]}')
    print(f'chi2/ndof: {m.fval / m.ndof}')


    return m



In [ ]:
m_pl_atlas = otimization(total_cost_pl_atlas,
                         mg_init = ensemble_parameters['atlas']['pl']['mg'], 
                         eps_init = ensemble_parameters['atlas']['pl']['epsilon'], 
                         a1_init = ensemble_parameters['atlas']['pl']['a1'], 
                         a2_init = ensemble_parameters['atlas']['pl']['a2'], 
                         model_type = 'pl', 
                         ensemble = 'atlas')


# mg: 0.23411637728741208 ± 0.00033574643816519965
# eps: 0.12685040781076112 ± 0.001340347140610439
# a1: 1.1692659320733771 ± 0.01474587351542503
# a2: 0.6767754519998356 ± 0.03910298646587622
# chi2/ndof: 350.38756456977086